# DeepAgent Quickstart — AWS Bedrock

Uses `deepagents` with a model served via **AWS Bedrock**.

### Prerequisites
- AWS credentials already configured (via `aws configure`, IAM role, or env vars)
- Bedrock model access enabled in your AWS account for your chosen model

```bash
# Optional — override region/profile via env vars
export AWS_DEFAULT_REGION=us-east-1
# export AWS_PROFILE=my-profile   # if using named profiles
```

In [ ]:
# Install dependencies (skip if already installed)
%pip install -q deepagents langchain-aws python-dotenv --prerelease=allow

In [1]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()  # loads from .env if present

# Verify AWS credentials are available
try:
    identity = boto3.client("sts").get_caller_identity()
    print(f"✅ AWS credentials OK — Account: {identity['Account']}, ARN: {identity['Arn']}")
except Exception as e:
    raise EnvironmentError(f"AWS credentials not configured: {e}")

✅ AWS credentials OK — Account: 575071290373, ARN: arn:aws:iam::575071290373:user/prashant_mudgal


In [2]:
from langchain_aws import ChatBedrock

# Choose the Bedrock model — change to any model you have access to
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

model = ChatBedrock(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    model_kwargs={
        "temperature": 0,
        "max_tokens": 4096,
    },
)

print(f"✅ Bedrock model ready: {BEDROCK_MODEL_ID} in {AWS_REGION}")

✅ Bedrock model ready: us.anthropic.claude-sonnet-4-5-20250929-v1:0 in us-east-1


In [3]:
# Define custom tools — plain Python functions with docstrings

def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    # Replace with a real weather API call in production
    return f"The weather in {city} is 22°C and sunny."


def list_s3_buckets() -> list[str]:
    """List all S3 bucket names in the configured AWS account."""
    s3 = boto3.client("s3")
    response = s3.list_buckets()
    return [b["Name"] for b in response.get("Buckets", [])]


def describe_bedrock_models(region: str = "us-east-1") -> list[dict]:
    """List all foundation models available in AWS Bedrock for a given region."""
    client = boto3.client("bedrock", region_name=region)
    response = client.list_foundation_models()
    return [
        {
            "modelId": m["modelId"],
            "providerName": m["providerName"],
            "modelName": m["modelName"],
        }
        for m in response.get("modelSummaries", [])
    ]


print("✅ Tools defined")

✅ Tools defined


In [4]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[get_weather, list_s3_buckets, describe_bedrock_models],
    system_prompt=(
        "You are a helpful AWS cloud assistant powered by Amazon Bedrock. "
        "Use your tools to answer questions about AWS resources and the environment."
    ),
)

print("✅ DeepAgent created")

✅ DeepAgent created


In [5]:
# --- Single invocation ---
result = agent.invoke(
    {"messages": [{"role": "user", "content": "How many S3 buckets do I have in my AWS account?"}]}
)

print(result["messages"][-1].content)

You have **213 S3 buckets** in your AWS account.


In [6]:
# --- Ask about available Bedrock models ---
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List all Anthropic models available in Bedrock in us-east-1."}]}
)

print(result["messages"][-1].content)

Here are all the Anthropic models available in Bedrock in us-east-1:

1. **Claude Sonnet 4** - `anthropic.claude-sonnet-4-20250514-v1:0`
2. **Claude Haiku 4.5** - `anthropic.claude-haiku-4-5-20251001-v1:0`
3. **Claude Fable 5** - `anthropic.claude-fable-5`
4. **Claude Sonnet 4.6** - `anthropic.claude-sonnet-4-6`
5. **Claude Opus 4.6** - `anthropic.claude-opus-4-6-v1`
6. **Claude Opus 4.8** - `anthropic.claude-opus-4-8`
7. **Claude Opus 4.7** - `anthropic.claude-opus-4-7`
8. **Claude Sonnet 4.5** - `anthropic.claude-sonnet-4-5-20250929-v1:0`
9. **Claude Opus 4.1** - `anthropic.claude-opus-4-1-20250805-v1:0`
10. **Claude Opus 4.5** - `anthropic.claude-opus-4-5-20251101-v1:0`
11. **Claude 3 Sonnet** - `anthropic.claude-3-sonnet-20240229-v1:0` (with variants: 28k, 200k context)
12. **Claude 3 Haiku** - `anthropic.claude-3-haiku-20240307-v1:0` (with variants: 48k, 200k context)

The models include the latest Claude 4 series (Opus, Sonnet, Haiku, and Fable) as well as the Claude 3 series mod